## Libraries

In [27]:
import numpy as np
import pandas as pd
import time
import random

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.neighbors import NearestNeighbors

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_

from statsmodels.stats.diagnostic import acorr_ljungbox

## Config

In [28]:
DATA_PATH = "../../data/full_data.xlsx"

TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

# temporal splits
TRAIN_END_DATE  = pd.Timestamp("2021-03-31")   # training period end
VAL_END_DATE    = pd.Timestamp("2022-03-31")   # val period end
TEST_START_DATE = pd.Timestamp("2022-04-01")   # test period start

# === Fill these with your best hyperparameters from tuning ===
WINDOW       = 12     # input length (months)
CNN_CHANNELS = 16     # conv filters
KERNEL_SIZE  = 3      # conv kernel size
LSTM_HIDDEN  = 64     # LSTM hidden units
DROPOUT      = 0.3
LR           = 5e-4
WEIGHT_DECAY = 1e-4

BATCH_SIZE    = 64
MAX_EPOCHS    = 80
PATIENCE      = 8
MAX_GRAD_NORM = 5.0

# device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# feature lists
continuous_cols = [
    "AvgNeighbourPrice_lag1",
    "local_I_lag1",
    "area_km2",
    "centroid_x",
    "centroid_y",
    "CoL_distance_km",
    "LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock_per_1000",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH",
    "LA_embed_0",
    "LA_embed_1",
    "LA_embed_2",
    "LA_embed_3",
    "LA_embed_4"
]

categorical_cols = [
    "LMIQuadrantlag1_2.0",
    "LMIQuadrantlag1_3.0",
    "LMIQuadrantlag1_4.0",
    "Region_East of England",
    "Region_London",
    "Region_North East",
    "Region_North West",
    "Region_South East",
    "Region_South West",
    "Region_West Midlands",
    "Region_Yorkshire and The Humber",
]

base_feature_cols = continuous_cols + categorical_cols

# reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


Using device: cuda


## Metric functions

In [29]:
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.mean(np.abs(y - yhat))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.sqrt(np.mean((y - yhat) ** 2))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return 100.0 * np.mean(2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps))

def mase(y, yhat, y_train, m=12, eps=1e-8):
    """Global MASE using in-sample seasonal naive with period m on y_train."""
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)
    if len(y_train) <= m:
        return np.nan
    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return np.mean(np.abs(y - yhat)) / scale

def directional_accuracy(df, entity_col, time_col, target_col, pred_col):
    """Fraction of times sign of month-on-month change is correct."""
    df = df.sort_values([entity_col, time_col]).copy()
    df["y_diff"] = df.groupby(entity_col)[target_col].diff()
    df["yhat_diff"] = df.groupby(entity_col)[pred_col].diff()
    mask = df["y_diff"].notna() & df["yhat_diff"].notna()
    same_dir = np.sign(df.loc[mask, "y_diff"]) == np.sign(df.loc[mask, "yhat_diff"])
    return same_dir.mean()

def growth_rate_error(df, entity_col, time_col, target_col, pred_col, m=12):
    """
    12-month growth rate error:
    g_t = (y_t - y_{t-m}) / y_{t-m}
    Returns MAE of growth-rate error.
    """
    df = df.sort_values([entity_col, time_col]).copy()
    df["y_lag_m"] = df.groupby(entity_col)[target_col].shift(m)
    df["yhat_lag_m"] = df.groupby(entity_col)[pred_col].shift(m)

    mask = df["y_lag_m"].notna() & df["yhat_lag_m"].notna() & (df["y_lag_m"] != 0)
    y_gr = (df.loc[mask, target_col] - df.loc[mask, "y_lag_m"]) / df.loc[mask, "y_lag_m"]
    yhat_gr = (df.loc[mask, pred_col] - df.loc[mask, "yhat_lag_m"]) / df.loc[mask, "yhat_lag_m"]

    return np.mean(np.abs(y_gr - yhat_gr))

def morans_i(
    residuals,
    xs,
    ys,
    k=5,
    eps=1e-8,
    symmetric=True,
    row_standardize=True,
    permutations=0,
    random_state=None,
):
    """
    Compute Moran's I for residuals using k-nearest neighbours
    with inverse-distance weights.
    """
    residuals = np.asarray(residuals, dtype=float)
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)

    N = len(residuals)
    if not (len(xs) == len(ys) == N):
        raise ValueError("residuals, xs, ys must all have the same length")

    # Center residuals
    x_mean = residuals.mean()
    x_dev = residuals - x_mean

    # Build kNN graph
    coords = np.column_stack([xs, ys])
    nbrs = NearestNeighbors(n_neighbors=k + 1).fit(coords)
    distances, indices = nbrs.kneighbors(coords)

    # Weight matrix W (dense; for large N you might switch to sparse)
    W = np.zeros((N, N), dtype=float)
    for i in range(N):
        neigh_idx = indices[i, 1:]          # skip self at index 0
        w = 1.0 / (distances[i, 1:] + eps)  # inverse-distance weights
        W[i, neigh_idx] = w

    # Optional symmetrisation
    if symmetric:
        W = 0.5 * (W + W.T)

    # Optional row standardisation
    if row_standardize:
        row_sums = W.sum(axis=1, keepdims=True)
        W = np.where(row_sums > 0, W / (row_sums + eps), 0.0)

    S0 = W.sum()

    # Moran's I numerator and denominator (vectorised)
    num = (W * (x_dev[:, None] * x_dev[None, :])).sum()
    den = (x_dev ** 2).sum() + eps

    I_obs = (N / S0) * (num / den)

    result = {
        "I": I_obs,
        "S0": S0,
        "permutations": None,
        "z_score": None,
        "p_value": None,
    }

    # Optional permutation test
    if permutations > 0:
        if isinstance(random_state, np.random.Generator):
            rng = random_state
        else:
            rng = np.random.default_rng(random_state)

        perm_I = np.empty(permutations, dtype=float)
        for b in range(permutations):
            perm = rng.permutation(x_dev)
            num_b = (W * (perm[:, None] * perm[None, :])).sum()
            perm_I[b] = (N / S0) * (num_b / den)

        mean_perm = perm_I.mean()
        std_perm = perm_I.std(ddof=1) + eps
        z = (I_obs - mean_perm) / std_perm

        extreme = np.sum(np.abs(perm_I - mean_perm) >= np.abs(I_obs - mean_perm))
        p_val = (extreme + 1) / (permutations + 1)

        result.update(
            {
                "permutations": perm_I,
                "z_score": z,
                "p_value": p_val,
            }
        )

    return result

## Load data and centroids

In [30]:
df_full = pd.read_excel(DATA_PATH, parse_dates=[TIME_COL])
df_full = df_full.query('Date < "2024-03-31"')
df_full = df_full.sort_values([ENTITY_COL, TIME_COL]).reset_index(drop=True)

# robust centroids (for Moran's I later)
df_full[["centroid_x", "centroid_y"]] = (
    df_full.groupby(ENTITY_COL)[["centroid_x", "centroid_y"]]
           .ffill()
           .bfill()
)

centroid_df = (
    df_full.drop_duplicates(ENTITY_COL)
           .set_index(ENTITY_COL)[["centroid_x", "centroid_y"]]
)

# main df: from 2007-04 onwards
df = df_full[df_full[TIME_COL] >= pd.Timestamp("2007-04-01")].copy()
df = df.sort_values([TIME_COL, ENTITY_COL]).reset_index(drop=True)

la_order = sorted(df[ENTITY_COL].unique())
N = len(la_order)
print("Number of LAs used:", N)

centroid_df = centroid_df.loc[la_order]
bad_las = centroid_df[centroid_df.isna().any(axis=1)].index.tolist()
if bad_las:
    print(f"⚠ Dropping {len(bad_las)} LAs with missing centroids:", bad_las)
    centroid_df = centroid_df.dropna()
    la_order = centroid_df.index.tolist()
    df = df[df[ENTITY_COL].isin(la_order)].copy()
    N = len(la_order)
    print("Updated number of LAs:", N)


Number of LAs used: 294


## Complete panel [T, M] and add price lags

In [31]:
df = (
    df.sort_values([TIME_COL, ENTITY_COL])
      .drop_duplicates(subset=[TIME_COL, ENTITY_COL], keep="last")
      .copy()
)

dates = pd.Index(sorted(df[TIME_COL].unique()))
T_total = len(dates)
print("Total time steps:", T_total)

full_index = pd.MultiIndex.from_product(
    [dates, la_order],
    names=[TIME_COL, ENTITY_COL]
)

feature_cols = base_feature_cols.copy()

df_panel = (
    df.set_index([TIME_COL, ENTITY_COL])
      .reindex(full_index)
      .sort_index()
)

# ffill/bfill features + target within each LA
df_panel[feature_cols + [TARGET_COL]] = (
    df_panel[feature_cols + [TARGET_COL]]
        .groupby(level=ENTITY_COL)
        .ffill()
        .bfill()
)

# ---- add price lags 1 and 12 ----
df_panel["price_lag1"] = (
    df_panel.groupby(level=ENTITY_COL)[TARGET_COL].shift(1)
)
df_panel["price_lag12"] = (
    df_panel.groupby(level=ENTITY_COL)[TARGET_COL].shift(12)
)

df_panel[["price_lag1", "price_lag12"]] = (
    df_panel[["price_lag1", "price_lag12"]]
        .groupby(level=ENTITY_COL)
        .ffill()
        .bfill()
)

lag_price_cols = ["price_lag1", "price_lag12"]
feature_cols = feature_cols + lag_price_cols

# last-resort fill
missing_total = df_panel[feature_cols + [TARGET_COL]].isna().sum().sum()
if missing_total > 0:
    print(f"⚠ {missing_total} NaNs after lag creation. Filling with column means.")
    col_means = df_panel[feature_cols + [TARGET_COL]].mean()
    df_panel[feature_cols + [TARGET_COL]] = df_panel[feature_cols + [TARGET_COL]].fillna(col_means)

print("NaNs after panel completion:",
      df_panel[feature_cols + [TARGET_COL]].isna().sum().sum())

F = len(feature_cols)

X_all = (
    df_panel[feature_cols]
    .to_numpy(dtype=np.float32)
    .reshape(T_total, N, F)
)
y_all = (
    df_panel[TARGET_COL]
    .to_numpy(dtype=np.float32)
    .reshape(T_total, N)
)

print("X_all shape:", X_all.shape)
print("y_all shape:", y_all.shape)

y_all_orig = y_all.copy()

Total time steps: 204
NaNs after panel completion: 0
X_all shape: (204, 294, 36)
y_all shape: (204, 294)


## Train val test time indices

In [32]:
train_end_idx  = np.searchsorted(dates, TRAIN_END_DATE,  side="right")
val_end_idx    = np.searchsorted(dates, VAL_END_DATE,    side="right")
test_start_idx = np.searchsorted(dates, TEST_START_DATE, side="left")

print(f"Train ends at idx {train_end_idx-1}, date {dates[train_end_idx-1].date()}")
print(f"Val   ends at idx {val_end_idx-1}, date {dates[val_end_idx-1].date()}")
print(f"Test starts at idx {test_start_idx}, date {dates[test_start_idx].date()}")


Train ends at idx 167, date 2021-03-01
Val   ends at idx 179, date 2022-03-01
Test starts at idx 180, date 2022-04-01


## Scaling

In [33]:
X_tv_flat = X_all[:val_end_idx].reshape(-1, F)
y_tv_flat = y_all[:val_end_idx].reshape(-1, 1)

x_scaler = StandardScaler()
X_all_scaled = X_all.copy()
X_all_scaled[:val_end_idx] = x_scaler.fit_transform(X_tv_flat).reshape(-1, N, F)
X_all_scaled[val_end_idx:] = x_scaler.transform(
    X_all[val_end_idx:].reshape(-1, F)
).reshape(-1, N, F)

y_scaler = RobustScaler()
y_all_scaled = y_all.copy()
y_all_scaled[:val_end_idx] = y_scaler.fit_transform(y_tv_flat).reshape(-1, N)
y_all_scaled[val_end_idx:] = y_scaler.transform(
    y_all[val_end_idx:].reshape(-1, 1)
).reshape(-1, N)

y_scale_factor = float(y_scaler.scale_[0])
print("NaNs in X_all_scaled:", np.isnan(X_all_scaled).sum())
print("NaNs in y_all_scaled:", np.isnan(y_all_scaled).sum())

NaNs in X_all_scaled: 0
NaNs in y_all_scaled: 0


## Dataset class

In [34]:
class PanelWindowDataset(Dataset):
    """
    Each sample is (X_seq, y_t, t_idx, n_idx):
      - X_seq: [window, F]
      - y_t: scalar
      - t_idx: time index
      - n_idx: LA index
    """
    def __init__(self, X_all, y_all, window, t_start, t_end):
        self.X_all = X_all
        self.y_all = y_all
        self.window = window
        self.T, self.N, self.F = X_all.shape

        indices = []
        for t in range(t_start, t_end):
            if t - window < 0:
                continue
            for n in range(self.N):
                indices.append((t, n))
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        t, n = self.indices[idx]
        X_seq = self.X_all[t - self.window:t, n, :]  # [window, F]
        y_t   = self.y_all[t, n]
        return (
            torch.tensor(X_seq, dtype=torch.float32),
            torch.tensor(y_t,   dtype=torch.float32),
            t,
            n
        )

train_ds = PanelWindowDataset(X_all_scaled, y_all_scaled,
                              window=WINDOW,
                              t_start=WINDOW,
                              t_end=train_end_idx)
val_ds   = PanelWindowDataset(X_all_scaled, y_all_scaled,
                              window=WINDOW,
                              t_start=train_end_idx,
                              t_end=val_end_idx)
test_ds  = PanelWindowDataset(X_all_scaled, y_all_scaled,
                              window=WINDOW,
                              t_start=test_start_idx,
                              t_end=T_total)

print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))
print("Test samples:", len(test_ds))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)


Train samples: 45864
Val samples: 3528
Test samples: 7056


## CNN-LSTM model

In [35]:
class CNNLSTM(nn.Module):
    def __init__(self, in_feats, cnn_channels, kernel_size, lstm_hidden, dropout=0.0):
        super().__init__()
        self.conv = nn.Conv1d(
            in_channels=in_feats,
            out_channels=cnn_channels,
            kernel_size=kernel_size,
            padding="same"
        )
        self.lstm = nn.LSTM(
            input_size=cnn_channels,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(lstm_hidden, 1)

    def forward(self, X_seq):
        """
        X_seq: [B, T, F]
        """
        B, T, F = X_seq.shape
        x = X_seq.permute(0, 2, 1)       # [B, F, T]
        x = self.conv(x)                 # [B, C_out, T]
        x = torch.relu(x)
        x = x.permute(0, 2, 1)           # [B, T, C_out]
        out, (h_n, c_n) = self.lstm(x)   # h_n: [1, B, H]
        h_last = h_n[-1]                 # [B, H]
        h_last = self.dropout(h_last)
        y_hat = self.fc(h_last).squeeze(-1)  # [B]
        return y_hat

## Train with early stopping

In [36]:
model = CNNLSTM(
    in_feats=F,
    cnn_channels=CNN_CHANNELS,
    kernel_size=KERNEL_SIZE,
    lstm_hidden=LSTM_HIDDEN,
    dropout=DROPOUT
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
loss_fn = nn.MSELoss()

best_val_mse = np.inf
best_epoch   = -1
epochs_no_improve = 0
best_state = None

print("\n=== Training final CNN-LSTM with early stopping ===")
start_time = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    train_losses = []

    for X_seq, y_t, t_idx, n_idx in train_loader:
        X_seq = X_seq.to(DEVICE)  # [B, T, F]
        y_t   = y_t.to(DEVICE)    # [B]

        optimizer.zero_grad()
        y_hat = model(X_seq)
        loss  = loss_fn(y_hat, y_t)

        if not torch.isfinite(loss):
            print(f"  ⚠ Non-finite training loss at epoch {epoch}. Aborting training.")
            train_losses = []
            break

        loss.backward()
        clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
        optimizer.step()
        train_losses.append(loss.item())

    if not train_losses:
        break

    # validation
    model.eval()
    val_losses = []
    with torch.no_grad():
        for X_seq, y_t, t_idx, n_idx in val_loader:
            X_seq = X_seq.to(DEVICE)
            y_t   = y_t.to(DEVICE)
            y_hat = model(X_seq)
            vloss = loss_fn(y_hat, y_t)
            if torch.isfinite(vloss):
                val_losses.append(vloss.item())

    if not val_losses:
        print("  ⚠ All val losses non-finite; stopping.")
        break

    val_mse = float(np.mean(val_losses))
    val_rmse_orig = np.sqrt(val_mse) * y_scale_factor

    print(f"Epoch {epoch:03d} | "
          f"train MSE={np.mean(train_losses):.4f} | "
          f"val MSE={val_mse:.4f} | "
          f"val RMSE(£)≈{val_rmse_orig:,.1f}")

    if val_mse + 1e-6 < best_val_mse:
        best_val_mse = val_mse
        best_epoch   = epoch
        epochs_no_improve = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

train_time = time.time() - start_time
print(f"Training finished in {train_time:.1f} seconds. Best epoch: {best_epoch}")

if best_state is not None:
    model.load_state_dict(best_state)
else:
    print("⚠ No best_state captured; using last epoch weights.")


=== Training final CNN-LSTM with early stopping ===
Epoch 001 | train MSE=0.0829 | val MSE=0.0270 | val RMSE(£)≈22,287.6
Epoch 002 | train MSE=0.0172 | val MSE=0.0148 | val RMSE(£)≈16,524.0
Epoch 003 | train MSE=0.0144 | val MSE=0.0283 | val RMSE(£)≈22,827.6
Epoch 004 | train MSE=0.0133 | val MSE=0.0188 | val RMSE(£)≈18,593.2
Epoch 005 | train MSE=0.0131 | val MSE=0.0241 | val RMSE(£)≈21,091.2
Epoch 006 | train MSE=0.0119 | val MSE=0.0246 | val RMSE(£)≈21,312.1
Epoch 007 | train MSE=0.0121 | val MSE=0.0161 | val RMSE(£)≈17,235.5
Epoch 008 | train MSE=0.0115 | val MSE=0.0229 | val RMSE(£)≈20,528.4
Epoch 009 | train MSE=0.0113 | val MSE=0.0148 | val RMSE(£)≈16,488.8
Epoch 010 | train MSE=0.0115 | val MSE=0.0158 | val RMSE(£)≈17,086.4
Epoch 011 | train MSE=0.0115 | val MSE=0.0121 | val RMSE(£)≈14,920.2
Epoch 012 | train MSE=0.0117 | val MSE=0.0256 | val RMSE(£)≈21,737.6
Epoch 013 | train MSE=0.0115 | val MSE=0.0311 | val RMSE(£)≈23,937.5
Epoch 014 | train MSE=0.0111 | val MSE=0.0157 | va

## Test evaluation

In [41]:
model.eval()
# we’ll fill arrays [T, N] for test period
y_pred_full_scaled = np.full_like(y_all_scaled, np.nan, dtype=np.float32)

with torch.no_grad():
    for X_seq, y_t, t_idx, n_idx in test_loader:
        X_seq = X_seq.to(DEVICE)
        y_hat = model(X_seq).cpu().numpy()  # [B]

        t_idx = np.array(t_idx)
        n_idx = np.array(n_idx)

        for i in range(len(t_idx)):
            t = t_idx[i]
            n = n_idx[i]
            y_pred_full_scaled[t, n] = y_hat[i]

# flatten test period arrays
test_mask = np.zeros((T_total, N), dtype=bool)
test_mask[test_start_idx:T_total, :] = True

y_true_test_orig = y_all_orig[test_mask]   # [S]
y_pred_test_scaled = y_pred_full_scaled[test_mask].reshape(-1, 1)

# inverse-transform predictions
y_pred_test_orig = y_scaler.inverse_transform(y_pred_test_scaled).ravel()

# build df_test with one row per (time, LA) for which we have predictions
t_coords, n_coords = np.where(test_mask)
dates_rep = dates[t_coords]
la_rep    = np.array(la_order)[n_coords]

df_test = pd.DataFrame({
    TIME_COL: dates_rep,
    ENTITY_COL: la_rep,
    "y_true": y_true_test_orig,
    "y_pred": y_pred_test_orig,
})
df_test["resid"] = df_test["y_true"] - df_test["y_pred"]

# For MASE scaling: use all pre-test y in original scale (train+val)
y_train_for_mase = y_all_orig[:val_end_idx].reshape(-1)


## Metrics

In [42]:
# =========================================================
# GLOBAL METRICS
# =========================================================
global_mae   = mae(df_test["y_true"], df_test["y_pred"])
global_rmse  = rmse(df_test["y_true"], df_test["y_pred"])
global_smape = smape(df_test["y_true"], df_test["y_pred"])
global_mase  = mase(df_test["y_true"], df_test["y_pred"], y_train_for_mase, m=12)

print("\n=== Global accuracy ===")
print(f"MAE   : {global_mae:,.3f}")
print(f"RMSE  : {global_rmse:,.3f}")
print(f"sMAPE : {global_smape:.3f}%")
print(f"MASE  : {global_mase:.3f}")

# =========================================================
# ACROSS-LA CONSISTENCY
# =========================================================
la_mae = df_test.groupby(ENTITY_COL).apply(lambda g: mae(g["y_true"], g["y_pred"]))
median_mae = float(la_mae.median())
p75_mae    = float(la_mae.quantile(0.75))

print("\n=== Across-LA consistency ===")
print(f"Median LA MAE       : {median_mae:,.3f}")
print(f"75th percentile MAE : {p75_mae:,.3f}")

# =========================================================
# SPATIO-TEMPORAL DIAGNOSTICS
# =========================================================
# Moran's I: mean residual per LA over test period

df_test = df_test.merge(
    centroid_df,
    left_on=ENTITY_COL,
    right_index=True,
    how="left"
)

la_resid_mean = df_test.groupby(ENTITY_COL)["resid"].mean()

centroids = (
    df_test
    .dropna(subset=["centroid_x", "centroid_y"])
    .sort_values(TIME_COL)
    .groupby(ENTITY_COL)
    .tail(1)
    .set_index(ENTITY_COL)[["centroid_x", "centroid_y"]]
)

# align
common = la_resid_mean.index.intersection(centroids.index)
la_resid_mean = la_resid_mean.loc[common]
centroids = centroids.loc[common]

mask_valid = centroids[["centroid_x", "centroid_y"]].notna().all(axis=1)
centroids_valid = centroids.loc[mask_valid]
la_resid_mean_valid = la_resid_mean.loc[centroids_valid.index]

print(f"\nLAs used for Moran's I: {len(centroids_valid)} / {len(la_resid_mean)}")

if len(centroids_valid) <= 1:
    I_moran = {"I": np.nan, "z_score": np.nan, "p_value": np.nan}
else:
    k_eff = min(5, len(centroids_valid) - 1)
    I_moran = morans_i(
        residuals=la_resid_mean_valid.values,
        xs=centroids_valid["centroid_x"].values,
        ys=centroids_valid["centroid_y"].values,
        k=k_eff,
        permutations=999,
        random_state=42,
    )
    print("\n=== Spatio-temporal diagnostics ===")
    print(f"Moran's I (mean residuals across LAs): {I_moran['I']:.4f}")
    print(f"Moran's I z score: {I_moran['z_score']:.4f}")
    print(f"Moran's I p value: {I_moran['p_value']:.4f}")

# Ljung–Box on monthly mean residuals (aggregate across LAs)
monthly_resid = df_test.groupby(TIME_COL)["resid"].mean().sort_index()
lb_res = acorr_ljungbox(monthly_resid, lags=[12], return_df=True)
q_stat = float(lb_res["lb_stat"].iloc[0])
p_val  = float(lb_res["lb_pvalue"].iloc[0])
print(f"Ljung–Box Q(12): stat={q_stat:.3f}, p={p_val:.4f}")

# =========================================================
# OPTIONAL: DIRECTIONAL ACCURACY & GROWTH-RATE ERROR
# =========================================================
dir_acc = directional_accuracy(df_test, ENTITY_COL, TIME_COL, "y_true", "y_pred")
gre_mae = growth_rate_error(df_test, ENTITY_COL, TIME_COL, "y_true", "y_pred", m=12)

print("\n=== Direction & growth ===")
print(f"Directional accuracy (MoM sign)   : {dir_acc:.3f}")
print(f"Growth-rate error MAE (12-month)  : {gre_mae:.4f}")


=== Global accuracy ===
MAE   : 29,946.580
RMSE  : 36,429.527
sMAPE : 9.243%
MASE  : 0.292

=== Across-LA consistency ===
Median LA MAE       : 27,625.359
75th percentile MAE : 42,778.757

LAs used for Moran's I: 294 / 294

=== Spatio-temporal diagnostics ===
Moran's I (mean residuals across LAs): 0.7101
Moran's I z score: 19.0748
Moran's I p value: 0.0010
Ljung–Box Q(12): stat=52.225, p=0.0000

=== Direction & growth ===
Directional accuracy (MoM sign)   : 0.426
Growth-rate error MAE (12-month)  : 0.0575


C:\Users\slong\AppData\Local\Temp\ipykernel_13088\3116216332.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  la_mae = df_test.groupby(ENTITY_COL).apply(lambda g: mae(g["y_true"], g["y_pred"]))


In [49]:

output_path = "../../results/cnn_lstm_final_test_results.xlsx"

summary_df = pd.DataFrame([{
    "model_type": "CNNLSTM",

    # --- hyperparameters ---
    "WINDOW": WINDOW,
    "CNN_CHANNELS": CNN_CHANNELS,
    "KERNEL_SIZE": KERNEL_SIZE,
    "LSTM_HIDDEN": LSTM_HIDDEN,
    "DROPOUT": DROPOUT,
    "LR": LR,
    "WEIGHT_DECAY": WEIGHT_DECAY,

    "MAE": global_mae,
    "RMSE": global_rmse,
    "sMAPE": global_smape,
    "MASE": global_mase,
    "Median_LA_MAE": median_mae,
    "P75_LA_MAE": p75_mae,
    "Morans_I": float(I_moran["I"]) if isinstance(I_moran, dict) else np.nan,
    "Morans_I_z": float(I_moran["z_score"]) if isinstance(I_moran, dict) and I_moran.get("z_score") is not None else np.nan,
    "Morans_I_p": float(I_moran["p_value"]) if isinstance(I_moran, dict) and I_moran.get("p_value") is not None else np.nan,
    "LjungBox_Q12": q_stat,
    "LjungBox_p": p_val,
    "Directional_Accuracy": dir_acc,
    "GrowthRateError_MAE": gre_mae

    }])

la_mae_df = la_mae.reset_index()
la_mae_df.columns = [ENTITY_COL, "LA_MAE"]


with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    summary_df.to_excel(writer, sheet_name="Global_Summary", index=False)
    la_mae_df.to_excel(writer, sheet_name="LA_MAE", index=False)
    df_test[[ENTITY_COL, TIME_COL, "y_true", "y_pred", "resid"]].to_excel(
        writer, sheet_name="Test_Predictions", index=False
    )

print(f"\nResults saved to: {output_path}")






Results saved to: ../../results/cnn_lstm_final_test_results.xlsx


In [46]:
la_mae

AreaCode
E06000001     7260.556152
E06000002     7569.358887
E06000003     9524.698242
E06000004    11588.534180
E06000005     7330.158203
                 ...     
E09000029    38811.300781
E09000030    26076.634766
E09000031    58218.527344
E09000032    55315.722656
E09000033    56285.750000
Length: 294, dtype: float32